<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# Avalon-MM Bus Helpers

This tutorial covers host-side register access, memory-side slave models, transaction monitoring, and the pyuvm Avalon-MM agent. Simulation examples require a cocotb DUT.

## 1. Bus and master BFM

`AvalonMMBus` groups an address signal with optional read, write, data, wait, response, byte-enable, and burst signals. `from_prefix()` binds standard `<prefix>_<signal>` names. Read-only and write-only ports are supported because only `address` is universally required.

`AvalonMMMasterBFM` is a lightweight cocotb host for single-beat register access. It waits for `waitrequest`, optionally waits for `readdatavalid`, validates signal widths, and supports access timeouts.

In [ ]:
import cocotb
from cocotb.clock import Clock
from cocotb.triggers import RisingEdge

from fpga_verification.sim.buses import AvalonMMBus, AvalonMMMasterBFM


@cocotb.test()
async def control_register_test(dut):
    cocotb.start_soon(Clock(dut.clk, 10, units="ns").start())

    avmm = AvalonMMMasterBFM.from_prefix(
        dut,
        "control",
        dut.clk,
        reset=dut.reset,
        default_byteenable=0xF,
    )
    avmm.start()

    dut.reset.value = 1
    await RisingEdge(dut.clk)
    dut.reset.value = 0
    await avmm.wait_reset_release(active_value=1)

    await avmm.write(0x00, 0x00000001, timeout_cycles=32)
    status = await avmm.read(0x04, timeout_cycles=32)
    await avmm.wait_set(0x04, 0x1, timeout_cycles=256)

    assert status & 0x1 in (0, 1)

In [ ]:
# Manual bus construction is useful when DUT signal names do not share a prefix.
bus = AvalonMMBus(
    address=dut.ctrl_address,
    writedata=dut.ctrl_writedata,
    write=dut.ctrl_write,
    read=dut.ctrl_read,
    readdata=dut.ctrl_readdata,
    waitrequest=getattr(dut, "ctrl_waitrequest", None),
    readdatavalid=getattr(dut, "ctrl_readdatavalid", None),
    byteenable=getattr(dut, "ctrl_byteenable", None),
)
avmm = AvalonMMMasterBFM(bus, dut.clk, reset=dut.reset)

Supported optional Avalon-MM signals: `waitrequest`, `readdatavalid`, and `byteenable`. The helper does not issue bursts or multiple outstanding reads.

## 2. Master convenience operations

In addition to `read()` and `write()`, the master provides common register operations:

- `read_modify_write()` applies a callback and returns `(old, new)`;
- `poll()` reads until a predicate succeeds;
- `wait_set()` waits until every selected bit is one;
- `wait_clear()` waits until every selected bit is zero;
- `set_packet_logging()` enables compact access summaries.

Always provide finite timeouts in regressions so a stalled interface produces a useful failure instead of hanging the test.

In [ ]:
async def configure_component(avmm):
    old, new = await avmm.read_modify_write(
        0x00,
        lambda value: value | 0x1,
        timeout_cycles=32,
    )
    status = await avmm.poll(
        0x04,
        lambda value: value & 0x1,
        interval_cycles=2,
        timeout_cycles=256,
    )
    return old, new, status

## 3. Slave and memory BFMs

`AvalonMMSlaveBFM` responds to a DUT master. Subclass it and implement `read_word()` and `write_word()` to model registers or custom address spaces. Unlike the simple host master, the slave accepts burstcount, byteenable, randomized waitrequest backpressure, configurable read latency, and queued in-order read responses.

`AvalonMMMemoryBFM` is the ready-made subclass for a byte-addressed object with `read(address, length)` and `write(address, data)` methods. `SparseByteMemory` from the DMA helpers satisfies that contract.

In [ ]:
from fpga_verification.sim.bfms import SparseByteMemory
from fpga_verification.sim.buses import (
    AvalonMMMemoryBFM,
    AvalonMMSlaveBFM,
    AvalonMMTransaction,
)


def make_memory_slave(dut):
    memory = SparseByteMemory()
    memory.write(0x1000, bytes.fromhex("44332211"))
    slave = AvalonMMMemoryBFM.from_prefix(
        dut,
        "memory",
        dut.clk,
        reset=dut.reset,
        memory=memory,
        byteorder="little",
        read_latency=2,
        record_transactions=True,
        randomize=True,
    ).start()
    return memory, slave

The memory model applies `byteenable` per byte lane. With little-endian ordering, lane 0 is the lowest addressed byte. `read_transactions` and `write_transactions` contain `AvalonMMTransaction` records when `record_transactions=True`; each record includes kind, address, data, byteenable, burstcount, and beat index.

## 4. AvalonMMMonitor and AvalonMMAgent

`AvalonMMMonitor` is a passive pyuvm component. It samples accepted read and write beats and publishes `AvalonMMTransaction` objects through its analysis port. `AvalonMMAgent` always owns a monitor and exposes the same port for connection to a scoreboard or coverage collector. In active mode it also owns an `AvalonMMMasterBFM`; in passive mode it only observes existing traffic.

In [ ]:
from fpga_verification.sim.agents import AvalonMMAgent, AvalonMMMonitor


def make_mm_agent(parent, dut):
    return AvalonMMAgent(
        "control_agent",
        parent,
        bus=AvalonMMBus.from_prefix(dut, "control"),
        clock=dut.clk,
        reset=dut.reset,
        packet_logging=True,
    )

# Connect agent.analysis_port to a pyuvm analysis export in connect_phase.

## 5. Lifecycle and limitations

Call `start()` on a master before access. A slave `start()` initializes outputs and launches its response coroutine; `init_idle()` only initializes outputs, and `stop()` cancels the task. Slave `pause`, `set_pause_generator()`, and `set_randomize()` control waitrequest backpressure.

The host master intentionally issues one transaction at a time and does not create bursts or multiple outstanding reads. The slave side can model bursts generated by a DUT. Response codes and write responses are driven to successful defaults but are not yet a complete Avalon-MM error-response model.